# Scaffolds from time series (Julia)

Computes H1 homological scaffolds (`freq` and `pers` files) with Ripserer for:
- HCP 1200 subjects, Schaefer300, `NoGSR_LowPassOnly`
- HCP 100 unrelated: Schaefer300, Glasser379, AAL89, Shen268, Schaefer100

Data locations: set `TOPOFINGER_HCP_ROOT` / `TOPOFINGER_HCP100_ROOT` (defaults match `code/topofinger/config.py`).
Filtering variants of the 1200 subjects are in `scripts/03_scaffolds_filtering_1200.jl`; sliding windows in `scripts/02_scaffolds_windows_1200.jl`.

In [ ]:
using DelimitedFiles
using Statistics
using Ripserer
using ProgressMeter
using NPZ
using MAT

# Data locations (override with environment variables)
HCP1200 = get(ENV, "TOPOFINGER_HCP_ROOT", "./data/HCP/1200subjects/schaefer300")
HCP100 = get(ENV, "TOPOFINGER_HCP100_ROOT", "./data/HCP/100unrelated")


# 1200 subjects

In [ ]:
data_dir = joinpath(HCP1200, "NoGSR_LowPassOnly") * "/"
data_dir_ts = joinpath(data_dir, "ts")
woutput_dir = joinpath(data_dir, "scaffolds", "whole")
# Create output directories if they don't exist
mkpath(joinpath(woutput_dir, "freq"))
mkpath(joinpath(woutput_dir, "pers"))

# Parameters
lr = "LR"
atlas = "Schaefer300"
gsr = "NoGSR"
bandpass = "LowPassOnly"

files = readdir(data_dir_ts)

# Extract unique subject IDs (first element of filename split by '_')
subjects = sort(collect(Set(split(f, '_')[1] for f in files)))

n = length(subjects)
p = ProgressMeter.Progress(n)

Threads.@threads for sub in subjects
    # Find matching files for both sessions
    sess_files = String[]
    valid_subject = true
    for n_sess in (1, 2)
        pattern_base = "$(sub)_rfMRI_REST$(n_sess)_$(lr)_$(atlas)+Tian_$(gsr)_$(bandpass)"
        found_idx = findfirst(f -> startswith(f, pattern_base), files)
        if found_idx === nothing
            valid_subject = false
            break
        end
        push!(sess_files, files[found_idx])
    end

    if !valid_subject
        ProgressMeter.next!(p)
        continue
    end

    for (n_sess, fname) in enumerate(sess_files)
        session_idx = n_sess - 1
        file_path = joinpath(data_dir_ts, fname)

        out_freq_path = joinpath(woutput_dir, "freq", "$(sub)_s$(session_idx).npz")
        out_pers_path = joinpath(woutput_dir, "pers", "$(sub)_s$(session_idx).npz")

        # Skip if outputs already exist
        if isfile(out_freq_path) && isfile(out_pers_path)
            continue
        end

        # Load data
        data = npzread(file_path)
        ts = data["timeseries"]'

        # Compute correlation matrix
        pms = cor(ts)
        num_regions = size(pms, 2)

        # Distance correlation
        dist_corrs = 1 .- pms
        dist_corrs = (dist_corrs .+ transpose(dist_corrs)) / 2

        d = ripserer(dist_corrs; reps=1, alg=:homology, verbose=false)

        p_scaffold_b = zeros(num_regions, num_regions)
        f_scaffold_b = zeros(num_regions, num_regions)

        for l in 1:length(d[2])
            g = d[2][l]
            pers = persistence(g)
            for rep in representative(g)
                i, j = vertices(rep)
                p_scaffold_b[i, j] += pers
                f_scaffold_b[i, j] += 1
                p_scaffold_b[j, i] += pers
                f_scaffold_b[j, i] += 1
            end
        end

        npzwrite(out_freq_path, f_scaffold_b)
        npzwrite(out_pers_path, p_scaffold_b)
    end

    ProgressMeter.next!(p)
end


# Schaefer300 (100 unrelated)

In [ ]:
data_dir = "$(HCP100)/schaefer300/no_hp/"
input_dir = "ts/LR/"
filenames = readdir("$data_dir$input_dir")

woutput_dir = "$(HCP100)/schaefer300/no_hp/scaffolds/whole"

n = 100
p = ProgressMeter.Progress(n)

Threads.@threads for filename in filenames
    sub_name = filename[1:6]
    session = filename[12]-1
    if isfile("$woutput_dir/freq/$(sub_name)_s$(session)") 
    else
        input_path = "$data_dir$input_dir$(filename)"
        #read CSV
        ts = readdlm(input_path,',', skipstart=1)
        #compute corr matris
        pms = cor(ts)
        num_regions = size(pms)[2]
        #copmute distance correlation
        dist_corrs = 1 .- pms
        #symmetrize matrix
        dist_corrs = (dist_corrs .+ transpose(dist_corrs))/2
        #dist_corrs = replace(dist_corrs, NaN => 1.5)  in case there is some missing series
        d = ripserer((dist_corrs .+ transpose(dist_corrs))/2; reps=1, alg=:homology, verbose=false)
        p_scaffold_b = zeros(num_regions,num_regions)
        f_scaffold_b = zeros(num_regions,num_regions)
        for l in 1:length(d[2])
            g = d[2][l]
            pers = persistence(g)
            for rep in representative(g) 
                i , j = vertices(rep)
                p_scaffold_b[i,j] += pers
                f_scaffold_b[i,j] += 1
                p_scaffold_b[j,i] += pers
                f_scaffold_b[j,i] += 1
            end
        end
        npzwrite("$woutput_dir/freq/$(sub_name)_s$(session)", f_scaffold_b)
        npzwrite("$woutput_dir/pers/$(sub_name)_s$(session)", p_scaffold_b)
    end
    ProgressMeter.next!(p)
end

# Glasser379

In [ ]:
#compute scaffold for whole correlations
data_dir = "$(HCP100)/glasser379/ts"
woutput_dir = "$(HCP100)/glasser379/scaffolds"

filenames = readdir(data_dir);

n = 100
p = ProgressMeter.Progress(n)
Threads.@threads for filename in filenames
    for sess in [1,2]
        #if isfile("$woutput_dir/freq/$(filename)_s$(sess-1)") 
        #elseif filename == "856766"
        #elseif filename == "101107"
        #else
            path = "$data_dir/$(filename)/rfMRI_REST$(sess)_LR/Glasser360/TS_Glasser360S_bp.mat"

            # Load the .mat file
            ts = matread(path)["TS"]
            pms = cor(transpose(ts))
            num_regions = size(pms)[2]
            dist_corrs = 1 .- pms
            dist_corrs = (dist_corrs .+ transpose(dist_corrs))/2
            dist_corrs = replace(dist_corrs, NaN => 1.5)
            #dist_corrs = Symmetric(dist_corrs)
            d = ripserer((dist_corrs .+ transpose(dist_corrs))/2; reps=1, alg=:homology, verbose=false)
            p_scaffold_b = zeros(num_regions,num_regions)
            f_scaffold_b = zeros(num_regions,num_regions)
            for l in 1:length(d[2])
                g = d[2][l]
                pers = persistence(g)
                for rep in representative(g) 
                    i , j = vertices(rep)
                    p_scaffold_b[i,j] += pers
                    f_scaffold_b[i,j] += 1
                    p_scaffold_b[j,i] += pers
                    f_scaffold_b[j,i] += 1
                end
            end
            npzwrite("$woutput_dir/whole/freq/$(filename)_s$(sess-1)", f_scaffold_b)
            npzwrite("$woutput_dir/whole/pers/$(filename)_s$(sess-1)", p_scaffold_b)
        #end
    end
    ProgressMeter.next!(p)
end

# AAL89

In [ ]:
data_dir = "$(HCP100)/aal89/"
input_dir = "ts/TimeSeriesAAL/"
woutput_dir = "$(HCP100)/aal89/scaffolds"
filenames = readdir("$data_dir$input_dir")

n = 100
p = ProgressMeter.Progress(n)

Threads.@threads for filename in filenames
    sub_name = filename[6:11]
    session = filename[23]
    if isfile("$woutput_dir/whole/freq/$(sub_name)_s$(session)")
    else
        input_path = "$data_dir$input_dir$(filename)"
        #read txt
        ts = readdlm(input_path)
        #compute corr matris
        pms = cor(ts)
        num_regions = size(pms)[2]
        #copmute distance correlation
        dist_corrs = 1 .- pms
        #symmetrize matrix
        dist_corrs = (dist_corrs .+ transpose(dist_corrs))/2
        #dist_corrs = replace(dist_corrs, NaN => 1.5)  in case there is some missing series
        d = ripserer((dist_corrs .+ transpose(dist_corrs))/2; reps=1, alg=:homology, verbose=false)
        p_scaffold_b = zeros(num_regions,num_regions)
        f_scaffold_b = zeros(num_regions,num_regions)
        for l in 1:length(d[2])
            g = d[2][l]
            pers = persistence(g)
            for rep in representative(g) 
                i , j = vertices(rep)
                p_scaffold_b[i,j] += pers
                f_scaffold_b[i,j] += 1
                p_scaffold_b[j,i] += pers
                f_scaffold_b[j,i] += 1
            end
        end
        npzwrite("$woutput_dir/whole/freq/$(sub_name)_s$(session)", f_scaffold_b)
        npzwrite("$woutput_dir/whole/pers/$(sub_name)_s$(session)", p_scaffold_b)
    end
    ProgressMeter.next!(p)
end

# Shen268

In [ ]:
data_dir = "$(HCP100)/shen268/"
input_dir = "ts/"
filenames = readdir("$data_dir$input_dir")

woutput_dir = "$(HCP100)/shen268/scaffolds/whole"

n = 100
p = ProgressMeter.Progress(n)

Threads.@threads for filename in filenames
    if isfile("$woutput_dir/freq/$(filename)_s0)") 
    else
        for session in [1,2]
            input_path = "$data_dir$input_dir$(filename)/$(filename)_REST$(session)_LR.npy"
            #read CSV
            ts= NPZ.npzread(input_path)
            #compute corr matris
            pms = cor(ts)
            num_regions = size(pms)[2]
            #copmute distance correlation
            dist_corrs = 1 .- pms
            #symmetrize matrix
            dist_corrs = (dist_corrs .+ transpose(dist_corrs))/2
            #dist_corrs = replace(dist_corrs, NaN => 1.5)  in case there is some missing series
            d = ripserer((dist_corrs .+ transpose(dist_corrs))/2; reps=1, alg=:homology, verbose=false)
            p_scaffold_b = zeros(num_regions,num_regions)
            f_scaffold_b = zeros(num_regions,num_regions)
            for l in 1:length(d[2])
                g = d[2][l]
                pers = persistence(g)
                for rep in representative(g) 
                    i , j = vertices(rep)
                    p_scaffold_b[i,j] += pers
                    f_scaffold_b[i,j] += 1
                    p_scaffold_b[j,i] += pers
                    f_scaffold_b[j,i] += 1
                end
            end
            npzwrite("$woutput_dir/freq/$(filename)_s$(session-1)", f_scaffold_b)
            npzwrite("$woutput_dir/pers/$(filename)_s$(session-1)", p_scaffold_b)
        end
    end
    ProgressMeter.next!(p)
end

# Schaefer100

In [ ]:
#compute scaffold for whole correlations
data_dir = "$(HCP100)/schaefer100/ts"
woutput_dir = "$(HCP100)/schaefer100/scaffolds"

filenames = readdir(data_dir);

n = 100
p = ProgressMeter.Progress(n)
Threads.@threads for filename in filenames
    for sess in [1,2]
        #if isfile("$woutput_dir/freq/$(filename)_s$(sess-1)") 
        #elseif filename == "856766"
        #elseif filename == "101107"
        #else
            path = "$data_dir/$(filename)/rfMRI_REST$(sess)_LR/Schaefer100/TS_Schaefer100S_gsr_bp_z.mat"

            # Load the .mat file
            ts = matread(path)["TS"]
            pms = cor(transpose(ts))
            num_regions = size(pms)[2]
            dist_corrs = 1 .- pms
            dist_corrs = (dist_corrs .+ transpose(dist_corrs))/2
            dist_corrs = replace(dist_corrs, NaN => 1.5)
            #dist_corrs = Symmetric(dist_corrs)
            d = ripserer((dist_corrs .+ transpose(dist_corrs))/2; reps=1, alg=:homology, verbose=false)
            p_scaffold_b = zeros(num_regions,num_regions)
            f_scaffold_b = zeros(num_regions,num_regions)
            for l in 1:length(d[2])
                g = d[2][l]
                pers = persistence(g)
                for rep in representative(g) 
                    i , j = vertices(rep)
                    p_scaffold_b[i,j] += pers
                    f_scaffold_b[i,j] += 1
                    p_scaffold_b[j,i] += pers
                    f_scaffold_b[j,i] += 1
                end
            end
            npzwrite("$woutput_dir/whole/freq/$(filename)_s$(sess-1)", f_scaffold_b)
            npzwrite("$woutput_dir/whole/pers/$(filename)_s$(sess-1)", p_scaffold_b)
        #end
    end
    ProgressMeter.next!(p)
end